# Protein ↔ ChemicalEntity Relation-Wise Merge

Merges Protein–Chemical triples from CrossBAR; resolves chemical tail names via
PubChem (IUPAC + SMILES) with a DrugBank fallback; deduplicates by `(head, relation, tail)`;
and saves the result.

## 0. Configuration

In [17]:
import os
import pandas as pd
import re

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PROTEIN_CHEMICALENTITY/ALL_PROTEIN_CHEMICALENTITY.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Chemical Lookup Dictionaries — PubChem and DrugBank

In [2]:
Pubchem = pd.read_pickle(DB_DIR + 'pubchem/combined_df.pkl')
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
del Pubchem

Drugbank = pd.read_csv(DB_DIR + 'drugbank/ALL_DRUGBANK_WITH_SMILE_PUBCHEM.csv')
Drugbank_dict = dict(zip(Drugbank['drugbank_id'], Drugbank['name']))

print(f"PubChem IUPAC dict: {len(Pubchem_IUPAC_CID_Dict):,} | DrugBank dict: {len(Drugbank_dict):,}")

PubChem IUPAC dict: 119,177,440 | DrugBank dict: 16,575


## 2. Load KG Sources

### 2a. CrossBAR

In [6]:
CrossBAR_Protein_CE = pd.read_csv(PROC_DIR + 'crossbar/Protein_ChemicalEntity_Crossbar.csv')
CrossBAR_Protein_CE.columns = CrossBAR_Protein_CE.columns.str.lower()
print(f"CrossBAR: {len(CrossBAR_Protein_CE):,} rows | columns: {list(CrossBAR_Protein_CE.columns)}")


CrossBAR_Protein_CE['kg_type'] = 'Generalised'
CrossBAR_Protein_CE['kg_source'] = 'CROssBAR'
CrossBAR_Protein_CE['species'] = 'Homo species'
CrossBAR_Protein_CE.head(2)

CrossBAR: 12 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_alt_id', 'head_detail_name', 'tail_alt_id', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,head_detail_name,tail_alt_id,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,A8MPY1,Protein_ChemicalEntity,3369,Protein,ChemicalEntity,CROssBAR,GABRR3,Gamma-aminobutyric acid receptor subunit rho-3,DB01567,Fludiazepam,Uniprot_ID,Pubchem,Generalised,Homo species
1,A1L3X4,Protein_ChemicalEntity,23954,Protein,ChemicalEntity,CROssBAR,MT1DP,Putative metallothionein MT1DP,DB12965,Silver,Uniprot_ID,Pubchem,Generalised,Homo species


## 3. Check and Fix Duplicate Columns

In [7]:
SOURCE_DFS = [('CrossBAR_Protein_CE', CrossBAR_Protein_CE)]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    print(f"[{name}] {'duplicate columns: ' + str(dupes) if dupes else '✓ no duplicates'}")

[CrossBAR_Protein_CE] ✓ no duplicates


In [8]:
CrossBAR_Protein_CE = CrossBAR_Protein_CE.loc[:, ~CrossBAR_Protein_CE.columns.duplicated(keep='first')]
print("[CrossBAR_Protein_CE] ✓ clean")

[CrossBAR_Protein_CE] ✓ clean


## 4. Align Schemas and Concatenate

In [10]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 12 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A8MPY1,Protein_ChemicalEntity,3369,Protein,<NA>,ChemicalEntity,CROssBAR,Uniprot_ID,Pubchem,Gamma-aminobutyric acid receptor subunit rho-3,Fludiazepam,Generalised,Homo species
1,A1L3X4,Protein_ChemicalEntity,23954,Protein,<NA>,ChemicalEntity,CROssBAR,Uniprot_ID,Pubchem,Putative metallothionein MT1DP,Silver,Generalised,Homo species


## 5. Standardise Fixed-Value Columns

In [11]:
final_df['head_type'] = 'Protein'
final_df['tail_type'] = 'ChemicalEntity'
final_df['relation']  = 'Protein_ChemicalEntity'

print("Unique relation:",  set(final_df['relation']))
print("Unique kg_source:", set(final_df['kg_source']))

Unique relation: {'Protein_ChemicalEntity'}
Unique kg_source: {'CROssBAR'}


In [12]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is', 'kg_type','species']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'Protein_ChemicalEntity'}
head_type           : {'Protein'}
tail_type           : {'ChemicalEntity'}
relation_type       : {<NA>}
kg_source           : {'CROssBAR'}
head_id_is          : {'Uniprot_ID'}
tail_id_is          : {'Pubchem'}
kg_type             : {'Generalised'}
species             : {'Homo species'}


## 6. Deduplicate

In [13]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
        'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 12 rows


## 7. QC — NaN Check

In [16]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,12,0,12
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 10. Save Output

In [19]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 12 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PROTEIN_CHEMICALENTITY/ALL_PROTEIN_CHEMICALENTITY.csv
